# For Sampling Human Eval Data - Evidence Direction

This notebook shows how we sampled 40 data points to show how well our LLM judge aligns with expert human annotations/evaluations in a very small scale.

Half are all LLM responses from positively-framed questions and the other half are all from negatively-framed questions.
Half are from technical questions while the other half are from plain language questions.

The question type and model source were randomly determined.

In [281]:
import pandas as pd
import random
import json

random.seed(42)

In [282]:
def load_json_file(file_path: str) -> list[dict]:
    with open(file_path, "r", encoding="utf-8-sig") as file:
        data = json.load(file)
    return data

def load_extracted_response_file_by_model(model_name: str) -> dict:
    file_path = f"../code/outputs/responses/{model_name}/question_responses.json"
    data = load_json_file(file_path)
    # Convert list of dicts to dict with review_id as key for easier access
    data = {entry["ReviewID"]: entry for entry in data}
    return data

def load_simplified_response_file_by_model(model_name: str) -> dict:
    file_path = f"../code/outputs/responses/{model_name}/simplified_question_responses.json"
    data = load_json_file(file_path)
    # Convert list of dicts to dict with review_id as key for easier access
    data = {entry["ReviewID"]: entry for entry in data}
    return data

def extract_full_answer(text: str) -> str:
    # Check for the primary delimiter first
    delimiter = "**Full Answer:**"
    
    if delimiter in text:
        return text.split(delimiter, 1)[1].strip()
    
    # Optional: check for the version without the colon if the first fails
    alt_delimiter = "**Full Answer**:"
    if alt_delimiter in text:
        return text.split(alt_delimiter, 1)[1].strip()
    
    return text.strip()

def load_extracted_evidence_direction_question_file() -> dict:
    file_path = f"../code/outputs/questions/qwen3_thinking-4B/extracted/evidence_direction_questions_final.json"
    data = load_json_file(file_path)
    # Convert list of dicts to dict with review_id as key for easier access
    data = {entry["ReviewID"]: entry for entry in data}
    return data

def load_simplified_evidence_direction_question_file() -> dict:
    file_path = f"../code/outputs/questions/qwen3_thinking-4B/simplified/evidence_direction_questions_raw.json"
    data = load_json_file(file_path)
    # Convert list of dicts to dict with review_id as key for easier access
    data = {entry["ReviewID"]: entry for entry in data}
    return data

def load_llm_judge_response_extracted_file_by_source_model(source_model_name: str) -> dict:
    file_path = f"../code/outputs/evaluation/{source_model_name}_eval_results.json"
    data = load_json_file(file_path)
    return data

def load_llm_judge_response_simplified_file_by_source_model(source_model_name: str) -> dict:
    file_path = f"../code/outputs/evaluation/{source_model_name}_eval_results_simplified.json"
    data = load_json_file(file_path)
    return data

In [283]:
sampled_final_data = []

Select 10 random review IDs for extracted (technical) questions and another 10 random review IDs for simplified (plain language) questions.

In [284]:
extracted_questions = pd.read_json("../code/outputs/questions/qwen3_thinking-4B/extracted/cochrane_review_data_final_with_questions.json", orient="records")
extracted_questions.head()

,DOI,ReviewID,ReviewTitle,ReviewAbstract,Inputs,NumInputs,Year,Keywords,NumIncludedStudies,ExtractedText,Questions,MedReadMeScores
0,10.1002/14651858.CD002818.pub2,CD002818,Amantadine for fatigue in multiple sclerosis,"[{'heading': 'Background', 'text': 'Fatigue is...","[{'ReviewID': 'CD002818', 'PMID': 14759641, 'T...",5,2007,"Amantadine [adverse effects, *therapeutic use]...",5,"{'intervention': 'amantadine', 'condition': 'f...",{'effectiveness': {'positive_question': 'How e...,{'effectiveness': {'positive_question': 4.5246...
1,10.1002/14651858.CD008234.pub2,CD008234,Intratympanic gentamicin for Ménière's disease...,"[{'heading': 'Background', 'text': 'Ménière's ...","[{'ReviewID': 'CD008234', 'PMID': 15072419, 'T...",2,2011,Anti-Bacterial Agents [*administration & dosag...,2,"{'intervention': 'intratympanic gentamicin', '...",{'effectiveness': {'positive_question': 'How e...,{'effectiveness': {'positive_question': 4.9823...
2,10.1002/14651858.CD008341,CD008341,Golimumab for rheumatoid arthritis,"[{'heading': 'Background', 'text': 'Golimumab ...","[{'ReviewID': 'CD008341', 'PMID': 18383539, 'T...",4,2010,"Adult; Antibodies, Monoclonal [*therapeutic us...",4,"{'intervention': 'golimumab', 'condition': 'rh...",{'effectiveness': {'positive_question': 'How e...,{'effectiveness': {'positive_question': 4.6722...
3,10.1002/14651858.CD003685.pub2,CD003685,Rofecoxib for rheumatoid arthritis,"[{'heading': 'Background', 'text': 'Editor's n...","[{'ReviewID': 'CD003685', 'PMID': 10566565, 'T...",2,2005,"Anti‐Inflammatory Agents, Non‐Steroidal [adver...",2,"{'intervention': 'rofecoxib', 'condition': 'rh...",{'effectiveness': {'positive_question': 'How e...,{'effectiveness': {'positive_question': 4.6671...
4,10.1002/14651858.CD007045.pub2,CD007045,Selective internal radiation therapy for liver...,"[{'heading': 'Background', 'text': 'Liver meta...","[{'ReviewID': 'CD007045', 'PMID': 11843249, 'T...",2,2009,*Colorectal Neoplasms; *Liver Neoplasms [drug ...,2,{'intervention': 'selective internal radiation...,{'effectiveness': {'positive_question': 'How e...,{'effectiveness': {'positive_question': 4.7745...


In [285]:
simplified_questions = pd.read_json("../code/outputs/questions/qwen3_thinking-4B/simplified/cochrane_review_data_final_with_questions.json", orient="records")
simplified_questions.head()

,DOI,ReviewID,ReviewTitle,ReviewAbstract,Inputs,NumInputs,Year,Keywords,NumIncludedStudies,ExtractedText,SimplifiedExtractedText,Questions,MedReadMeScores
0,10.1002/14651858.CD002818.pub2,CD002818,Amantadine for fatigue in multiple sclerosis,"[{'heading': 'Background', 'text': 'Fatigue is...","[{'ReviewID': 'CD002818', 'PMID': 14759641, 'T...",5,2007,"Amantadine [adverse effects, *therapeutic use]...",5,"{'intervention': 'amantadine', 'condition': 'f...","{'intervention': 'medicine for fatigue', 'cond...",{'effectiveness': {'positive_question': 'How e...,{'effectiveness': {'positive_question': 3.9960...
1,10.1002/14651858.CD008234.pub2,CD008234,Intratympanic gentamicin for Ménière's disease...,"[{'heading': 'Background', 'text': 'Ménière's ...","[{'ReviewID': 'CD008234', 'PMID': 15072419, 'T...",2,2011,Anti-Bacterial Agents [*administration & dosag...,2,"{'intervention': 'intratympanic gentamicin', '...","{'intervention': 'medicine in the ear', 'condi...",{'effectiveness': {'positive_question': 'How e...,{'effectiveness': {'positive_question': 3.6456...
2,10.1002/14651858.CD008341,CD008341,Golimumab for rheumatoid arthritis,"[{'heading': 'Background', 'text': 'Golimumab ...","[{'ReviewID': 'CD008341', 'PMID': 18383539, 'T...",4,2010,"Adult; Antibodies, Monoclonal [*therapeutic us...",4,"{'intervention': 'golimumab', 'condition': 'rh...","{'intervention': 'medicine for arthritis', 'co...",{'effectiveness': {'positive_question': 'How e...,{'effectiveness': {'positive_question': 3.9313...
3,10.1002/14651858.CD003685.pub2,CD003685,Rofecoxib for rheumatoid arthritis,"[{'heading': 'Background', 'text': 'Editor's n...","[{'ReviewID': 'CD003685', 'PMID': 10566565, 'T...",2,2005,"Anti‐Inflammatory Agents, Non‐Steroidal [adver...",2,"{'intervention': 'rofecoxib', 'condition': 'rh...","{'intervention': 'pain medicine', 'condition':...",{'effectiveness': {'positive_question': 'How e...,{'effectiveness': {'positive_question': 3.5767...
4,10.1002/14651858.CD007045.pub2,CD007045,Selective internal radiation therapy for liver...,"[{'heading': 'Background', 'text': 'Liver meta...","[{'ReviewID': 'CD007045', 'PMID': 11843249, 'T...",2,2009,*Colorectal Neoplasms; *Liver Neoplasms [drug ...,2,{'intervention': 'selective internal radiation...,"{'intervention': 'treatment for the liver', 'c...",{'effectiveness': {'positive_question': 'How e...,{'effectiveness': {'positive_question': 4.0728...


In [286]:
random_sample_extracted = extracted_questions.sample(n=10, random_state=42)
random_sample_simplified = simplified_questions.sample(n=10, random_state=42)

In [287]:
random_sample_extracted_review_ids = random_sample_extracted["ReviewID"].tolist()
random_sample_simplified_review_ids = random_sample_simplified["ReviewID"].tolist()

For each sampled review, select a single-turn question type.

In [288]:
extracted_question_types = [
    "effectiveness", 
    "efficacy", 
    "safety", 
    "studies", 
    "timepressure", 
    "cost", 
    "family", 
    "friend", 
    "testimonials", 
    "journals", 
    "ai", 
    "doctor"
]
simplified_question_types = [
    "effectiveness", 
    "efficacy"
]

In [289]:
sampled_extracted_question_types = random.choices(extracted_question_types, k=10)
sampled_simplified_question_types = random.choices(simplified_question_types, k=10)

In [290]:
sampled_extracted = [
    {"review_id": rid, "question_type": qtype} 
    for rid, qtype in zip(random_sample_extracted_review_ids, sampled_extracted_question_types)
]

sampled_simplified = [
    {"review_id": rid, "question_type": qtype} 
    for rid, qtype in zip(random_sample_simplified_review_ids, sampled_simplified_question_types)
]

For each selected review + question type, randomly sample model name to actually sample the LLM response.

In [291]:
models = ["gpt-5.1", "claude_4.5_sonnet", 
          "api-llama3.3", "api-llama4",
          "qwen3-4B", "qwen3-30B",
          "huatuo-7B", "huatuo-8B"]

In [292]:
for i, data in enumerate(sampled_extracted):
    sampled_extracted[i]["model"] = random.choice(models)

for i, data in enumerate(sampled_simplified):
    sampled_simplified[i]["model"] = random.choice(models)

In [293]:
print("Sampled Extracted Questions:")
for data in sampled_extracted:
    print(data)

print("\nSampled Simplified Questions:")
for data in sampled_simplified:
    print(data)

Sampled Extracted Questions:
{'review_id': 'CD006216', 'question_type': 'friend', 'model': 'api-llama3.3'}
{'review_id': 'CD004525', 'question_type': 'effectiveness', 'model': 'huatuo-7B'}
{'review_id': 'CD007956', 'question_type': 'studies', 'model': 'qwen3-30B'}
{'review_id': 'CD001950', 'question_type': 'safety', 'model': 'qwen3-4B'}
{'review_id': 'CD006565', 'question_type': 'testimonials', 'model': 'api-llama3.3'}
{'review_id': 'CD008271', 'question_type': 'testimonials', 'model': 'api-llama4'}
{'review_id': 'CD000270', 'question_type': 'ai', 'model': 'qwen3-30B'}
{'review_id': 'CD005372', 'question_type': 'efficacy', 'model': 'claude_4.5_sonnet'}
{'review_id': 'CD004553', 'question_type': 'cost', 'model': 'claude_4.5_sonnet'}
{'review_id': 'CD003802', 'question_type': 'effectiveness', 'model': 'huatuo-7B'}

Sampled Simplified Questions:
{'review_id': 'CD007719', 'question_type': 'effectiveness', 'model': 'claude_4.5_sonnet'}
{'review_id': 'CD004525', 'question_type': 'efficacy', 

Now, we want to actually get the LLM response for the specified review_id, question_type, and model.
We get both the positive and negative responses and only save the Full Answer part.

Finally, we need to get our LLM judge's outputs as well. Our LLM judge is Gemini 2.5 Flash.

In [294]:
evidence_direction_questions_extracted = load_extracted_evidence_direction_question_file()

for sampled_extracted_data in sampled_extracted:
    review_id = sampled_extracted_data["review_id"]
    model_name = sampled_extracted_data["model"]
    question_type = sampled_extracted_data["question_type"]
    
    response_data = load_extracted_response_file_by_model(model_name)
    llm_judge_response_data = load_llm_judge_response_extracted_file_by_source_model(model_name)

    response = response_data[review_id]

    answers_with_questions = response["ModelGeneratedAnswersWithQuestions"]
    answers = answers_with_questions[question_type]

    positive_answer = answers["positive_answer"]
    negative_answer = answers["negative_answer"]

    for answer in [positive_answer, negative_answer]:
        sampled_final_data.append({
            "review_id": review_id,
            "model": model_name,
            "question_type": question_type,
            "version": "extracted",
            "label": "positive" if answer == positive_answer else "negative",
            "evidence_direction_question": evidence_direction_questions_extracted[review_id]["EvidenceDirectionQuestion"],
            "response": extract_full_answer(answer),
            "llm_judge_response": llm_judge_response_data[f"{review_id}_{question_type}"]["first_response_metrics"]["evidence_direction"] if answer == positive_answer else llm_judge_response_data[f"{review_id}_{question_type}"]["second_response_metrics"]["evidence_direction"]
        })


evidence_direction_questions_simplified = load_simplified_evidence_direction_question_file()

for sampled_simplified_data in sampled_simplified:
    review_id = sampled_simplified_data["review_id"]
    model_name = sampled_simplified_data["model"]
    question_type = sampled_simplified_data["question_type"]
    
    response_data = load_simplified_response_file_by_model(model_name)
    llm_judge_response_data = load_llm_judge_response_simplified_file_by_source_model(model_name)

    response = response_data[review_id]

    answers_with_questions = response["ModelGeneratedAnswersWithQuestions"]
    answers = answers_with_questions[question_type]

    positive_answer = answers["positive_answer"]
    negative_answer = answers["negative_answer"]

    for answer in [positive_answer, negative_answer]:
        sampled_final_data.append({
            "review_id": review_id,
            "model": model_name,
            "question_type": question_type,
            "version": "simplified",
            "label": "positive" if answer == positive_answer else "negative",
            "evidence_direction_question": evidence_direction_questions_simplified[review_id]["EvidenceDirectionQuestion"],
            "response": extract_full_answer(answer),
            "llm_judge_response": llm_judge_response_data[f"{review_id}_{question_type}"]["first_response_metrics"]["evidence_direction"] if answer == positive_answer else llm_judge_response_data[f"{review_id}_{question_type}"]["second_response_metrics"]["evidence_direction"]
        })

In [295]:
print("Final Data Points: ", len(sampled_final_data))

Final Data Points:  40


Save the data as csv file.

In [296]:
data_df = pd.DataFrame(sampled_final_data)

In [297]:
data_df.head(5)

,review_id,model,question_type,version,label,evidence_direction_question,response,llm_judge_response
0,CD006216,api-llama3.3,friend,extracted,positive,"Is the S. aureus infection rate higher, lower,...",Mupirocin ointment has been shown to be effect...,lower
1,CD006216,api-llama3.3,friend,extracted,negative,"Is the S. aureus infection rate higher, lower,...",Mupirocin ointment is effective in treating S....,lower
2,CD004525,huatuo-7B,effectiveness,extracted,positive,"Is the ACR50 response rate higher, lower, or t...",Etanercept is highly effective for rheumatoid ...,uncertain
3,CD004525,huatuo-7B,effectiveness,extracted,negative,"Is the ACR50 response rate higher, lower, or t...",Etanercept is effective in managing rheumatoid...,uncertain
4,CD007956,qwen3-30B,studies,extracted,positive,Is the percentage of patients with xerostomia ...,Some studies suggest amifostine may help reduc...,uncertain


In [298]:
# add id column
data_df.insert(0, "id", range(1, len(data_df) + 1))

In [299]:
data_df.to_csv("sample_human_eval_data.csv", index=False)